In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, roc_auc_score
from scipy.ndimage.filters import uniform_filter1d
from keras.models import Sequential, Model
from keras.layers import Conv1D, MaxPool1D, Dense, Dropout, Flatten, \
BatchNormalization, Input, Activation
from tensorflow.keras.optimizers import Adam

In [ ]:
train_d = pd.read_csv("dataset/exoTrain.csv")

In [ ]:
train_d.head()

In [ ]:
train_d.shape

In [ ]:
train_d.isnull().sum()

In [ ]:
train_d['LABEL'].value_counts()

In [ ]:
categ = {2: 1, 1: 0}
train_d.LABEL = [categ[item] for item in train_d.LABEL]

In [ ]:
train_d['LABEL'].value_counts()

In [ ]:
sns.countplot(x=train_d['LABEL'])

In [ ]:
def stats_plots(train_d):
    means = train_d.mean(axis=1)
    medians = train_d.median(axis=1)
    std = train_d.std(axis=1)
    maxval = train_d.max(axis=1)
    minval = train_d.min(axis=1)
    skew = train_d.skew(axis=1)
    fig = plt.figure(figsize=(12,8))
    ax = fig.add_subplot(231)
    ax.hist(means,alpha=0.8,bins=50)
    ax.set_xlabel('Mean Intensity')
    ax.set_ylabel('Num. of Stars')
    ax = fig.add_subplot(232)
    ax.hist(medians,alpha=0.8,bins=50)
    ax.set_xlabel('Median Intensity')
    ax.set_ylabel('Num. of Stars')
    ax = fig.add_subplot(233)
    ax.hist(std,alpha=0.8,bins=50)
    ax.set_xlabel('Intensity Standard Deviation')
    ax.set_ylabel('Num. of Stars')
    ax = fig.add_subplot(234)
    ax.hist(maxval,alpha=0.8,bins=50)
    ax.set_xlabel('Maximum Intensity')
    ax.set_ylabel('Num. of Stars')
    ax = fig.add_subplot(235)
    ax.hist(minval,alpha=0.8,bins=50)
    ax.set_xlabel('Minimum Intensity')
    ax.set_ylabel('Num. of Stars')
    ax = fig.add_subplot(236)
    ax.hist(skew,alpha=0.8,bins=50)
    ax.set_xlabel('Intensity Skewness')
    ax.set_ylabel('Num. of Stars')

stats_plots(train_d)
plt.show()

In [ ]:
train_d.drop(train_d[train_d['FLUX.1']>250000].index, axis=0, inplace=True)

In [ ]:
x_train = train_d.drop(["LABEL"],axis=1)
y_train = train_d["LABEL"]

In [ ]:
from scipy import ndimage
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import recall_score, precision_score, classification_report,accuracy_score,confusion_matrix

In [ ]:
x_train = normalized = normalize(x_train)

In [ ]:
x_train = filtered = ndimage.filters.gaussian_filter(x_train, sigma=10)

In [ ]:
std_scaler = StandardScaler()
x_train = scaled = std_scaler.fit_transform(x_train)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA() 
x_train = pca.fit_transform(x_train)
x_train = pca.transform(x_train)
total=sum(pca.explained_variance_)
k=0
current_variance=0
while current_variance/total < 0.90:
    current_variance += pca.explained_variance_[k]
    k=k+1
print(k)

In [ ]:
pca = PCA(n_components=37)
x_train = pca.fit_transform(x_train)
plt.figure()
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Variance (%)') 
plt.title('Exoplanet Dataset Explained Variance')
plt.show()

In [ ]:
from imblearn.over_sampling import SMOTE
model = SMOTE()
ov_train_x,ov_train_y = model.fit_resample(train_d.drop('LABEL',axis=1), train_d['LABEL'])
ov_train_y = ov_train_y.astype('int')

ov_train_y.value_counts().reset_index().plot(kind='bar', x='index', y='LABEL')

In [ ]:
train_X, test_X, train_y, test_y = train_test_split(ov_train_x, ov_train_y, test_size=0.33, random_state=42)

In [ ]:
test_data = pd.read_csv("dataset/exoTest.csv")

In [ ]:
test_data.LABEL = [categ[item] for item in test_data.LABEL]

In [ ]:
train_X.shape

In [ ]:
test_data.shape

In [ ]:
test_X.shape

In [ ]:
n_features = train_X.shape[1]

In [ ]:
n_features

In [ ]:
import keras
from keras.layers import *
from keras.models import *
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, LeakyReLU
from tensorflow.keras.layers import Flatten, Activation, BatchNormalization
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras import layers
import tensorflow as tf

In [ ]:
train_X.shape

In [ ]:
train_X.shape[1:]

In [ ]:
model = Sequential()
model.add(Conv1D(filters=16, kernel_size=2, activation='relu', input_shape=(3197,1), kernel_regularizer='l2'))
model.add(Conv1D(filters=16, kernel_size=2, activation='relu'))
model.add(MaxPooling1D(pool_size=2, strides=2))
model.add(Dropout(0.2))
model.add(Flatten())
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.4))
model.add(Dense(48, activation="relu"))
model.add(Dense(18, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

In [ ]:
x_test, y_test = test_data.drop(columns = ['LABEL']), test_data.loc[:, 'LABEL']

In [ ]:
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.models import load_model
#from itertools import chain
model.compile(optimizer = Adam(4e-5), loss='binary_crossentropy', metrics=['accuracy'])

    # Fit model
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
history = model.fit(train_X, train_y, validation_split = 0.2, batch_size=128, callbacks=[early_stop], epochs=40, verbose=2)

    # Evaluate the model
train_acc = model.evaluate(train_X, train_y, verbose=2)
test_acc = model.evaluate(test_X, test_y, verbose=2)

In [ ]:
def prediction_metrics(y_test, y_pred, y_class_pred, matrix):
    FP = matrix[0][1] 
    FN = matrix[1][0]
    TP = matrix[1][1]
    TN = matrix[0][0]

    sens = TP/(TP+FN)
    spec = TN/(TN+FP) 
    g_mean = np.sqrt(sens * spec)

    accuracy = accuracy_score(y_test, y_class_pred)
    balanced_accuracy = balanced_accuracy_score(y_test, y_class_pred)
    precision = precision_score(y_test, y_class_pred)
    recall = recall_score(y_test, y_class_pred)
    f1 = f1_score(y_test, y_class_pred)
    auc = roc_auc_score(y_test, y_pred)

    print('\t\t Prediction Metrics\n')
    print("Accuracy:\t", "{:0.3f}".format(accuracy))
    print("Precision:\t", "{:0.3f}".format(precision))
    print("Recall:\t\t", "{:0.3f}".format(recall))
    print("\nF1 Score:\t", "{:0.3f}".format(f1))
    print("ROC AUC:\t", "{:0.3f}".format(auc))
    print("Balanced\nAccuracy:\t", "{:0.3f}".format(balanced_accuracy))
    print("\nSensitivity:\t", "{:0.3f}".format(sens))
    print("Specificity:\t", "{:0.3f}".format(spec))
    print("Geometric Mean:\t", "{:0.3f}".format(g_mean))

In [ ]:
def conf_matrix(y_test, y_pred):

    matrix = confusion_matrix(y_test, y_pred)
    df_cm = pd.DataFrame(matrix, columns=[0, 1], index = [0, 1])
    df_cm.index.name = 'Truth'
    df_cm.columns.name = 'Predicted'
    plt.figure(figsize = (10,7))
    sns.set(font_scale=1.4) 
    sns.heatmap(df_cm, cmap="BuGn", annot=True, annot_kws={"size": 16})
    plt.show()
    
    return matrix

In [ ]:
def graph_acc(history):
    # Plot loss during training
    plt.subplot(211)
    plt.title('Loss')
    plt.plot(history.history['loss'], label='train')
    plt.plot(history.history['val_loss'], label='test')
    plt.legend()

    # Plot accuracy during training
    plt.subplot(212)
    plt.title('Accuracy')
    plt.plot(history.history['accuracy'], label='train')
    plt.plot(history.history['val_accuracy'], label='test')
    plt.legend()
    plt.show()

In [ ]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [ ]:
y_class_pred = (model.predict(test_X) > 0.5).astype("int32")
y_pred = model.predict(test_X)

    # Accuracy graph
graph_acc(history)

    # Confustion matrix
matrix = conf_matrix(test_y, y_class_pred)

    # Metrics
prediction_metrics(test_y, y_pred, y_class_pred, matrix)

In [ ]:
y_class_pred = (model.predict(x_test) > 0.5).astype("int32")
y_pred = model.predict(x_test)

    # Accuracy graph
graph_acc(history)

    # Confustion matrix
matrix = conf_matrix(y_test, y_class_pred)

    # Metrics
prediction_metrics(y_test, y_pred, y_class_pred, matrix)

In [ ]:
_, accuracy = model.evaluate(x_test, y_test, batch_size=2048, verbose=0)

In [ ]:
_, ay = model.evaluate(test_X, test_y, batch_size=2048, verbose=0)

In [ ]:
ay

In [ ]:
accuracy

In [ ]:
score = accuracy * 100.0
score

In [ ]:
test_y.shape

In [ ]:
pred = model.predict(x_test)

In [ ]:
pred

In [ ]:
y_test

### using ml models

In [ ]:
def mmodel(classifier,dtrain_x,dtrain_y,dtest_x,dtest_y):
    classifier.fit(dtrain_x,dtrain_y)

    prediction=classifier.predict(dtest_x)

    print('Validation accuracy of model is', accuracy_score(prediction,dtest_y))
    print ("\nClassification report :\n",(classification_report(dtest_y,prediction)))

    plt.figure(figsize=(13,10))
    plt.subplot(221)
    sns.heatmap(confusion_matrix(dtest_y,prediction),annot=True,cmap="viridis",fmt = "d",linecolor="k",linewidths=3)
    plt.title("CONFUSION MATRIX",fontsize=20)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import RandomOverSampler

In [ ]:
from sklearn.tree import DecisionTreeClassifier
ds_model = DecisionTreeClassifier(max_depth=12, random_state=13)
mmodel(ds_model,train_X,train_y,test_X,test_y)

In [ ]:
mmodel(RandomForestClassifier(), train_X, train_y, test_X, test_y)

In [ ]:
from xgboost import XGBClassifier

xgb_classifier = XGBClassifier()
xgb_classifier.fit(train_X, train_y)
y_pred_xgb = xgb_classifier.predict(test_X)
accuracy_xgb = accuracy_score(test_y, y_pred_xgb)
print(accuracy_xgb)